# Weight Divergence Analysis in Federated Learning

## Overview

This notebook analyzes **model weight divergence** across clients and communication rounds, comparing IID vs Non-IID data partitions.

**Goal**: Understand how client models diverge in their learned weights and how aggregation affects convergence.

**Key Metrics**:
- Cosine similarity between local and global models
- L2 distance to global model
- Model consensus distance
- Outlier client identification
- Branch-specific weight divergence (Detail vs Semantic)

## Section 1: Environment Setup

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple

# Add analysis utilities to path
sys.path.insert(0, '/home/moustafa/Me/Projects/Grad/Code/BiseNet-FL')

# Import custom modules
from fl_cityscapes_bisenetv2.lib.checkpoint_utils import (
    load_local_model, load_global_model, get_available_rounds, 
    get_clients_in_round, load_client_metadata
)
from fl_cityscapes_bisenetv2.analysis.weight_divergence.weight_utils import (
    extract_weights_by_layer, compute_layer_wise_cosine_similarity,
    compute_layer_wise_l2_distance, get_branch_state_dict,
    map_layer_to_branch
)
from fl_cityscapes_bisenetv2.analysis.weight_divergence.model_similarity import (
    cosine_similarity_matrices, l2_distance_matrix,
    identify_outlier_clients, compute_model_consensus_distance,
    branch_similarity_matrices
)

print("✓ All imports successful")

## Section 2: Configuration

In [ ]:
# Configuration
PARTITIONS = ['iid_partitions', 'non_iid_partitions']
BASE_PATH = 'res'
NUM_CLASSES = 19
BRANCH_NAMES = ['detail_branch', 'semantic_branch', 'decoder']

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

# Verify checkpoint directories exist
print("Checking checkpoint directories...")
for partition in PARTITIONS:
    local_path = os.path.join(BASE_PATH, partition, 'local_models')
    global_path = os.path.join(BASE_PATH, partition, 'global_models')
    print(f"  {partition}:")
    print(f"    Local models: {os.path.exists(local_path)}")
    print(f"    Global models: {os.path.exists(global_path)}")

print("\n✓ Configuration complete")

## Section 3: Load Models & Extract Weights

In [ ]:
print("Loading models and computing weight metrics...")

# Store data structures
weight_analysis = {partition: {} for partition in PARTITIONS}
similarity_matrices = {partition: {} for partition in PARTITIONS}
consensus_distances = {partition: {} for partition in PARTITIONS}
outlier_clients = {partition: {} for partition in PARTITIONS}

for partition_name in PARTITIONS:
    print(f"\n=== Processing {partition_name} ===")
    
    # Get available rounds
    available_rounds = get_available_rounds(BASE_PATH, partition_name)
    print(f"  Available rounds: {available_rounds}")
    
    if not available_rounds:
        print(f"  No rounds found for {partition_name}, skipping...")
        continue
    
    for round_num in sorted(available_rounds):
        print(f"  Round {round_num}...", end=' ')
        
        # Load global model
        global_state = load_global_model(BASE_PATH, partition_name, round_num)
        if global_state is None:
            print("SKIP (no global model)")
            continue
        
        # Get clients in this round
        clients = get_clients_in_round(BASE_PATH, partition_name, round_num)
        if not clients:
            print("SKIP (no clients)")
            continue
        
        round_models = {}
        
        # Load local models
        for client_id in clients:
            local_state = load_local_model(BASE_PATH, partition_name, round_num, client_id)
            if local_state is not None:
                round_models[client_id] = local_state
        
        if not round_models:
            print("SKIP (no local models)")
            continue
        
        # Compute similarity matrices
        try:
            sim_matrix, client_ids = cosine_similarity_matrices(round_models)
            similarity_matrices[partition_name][round_num] = (sim_matrix, client_ids)
            
            # Compute consensus distance
            consensus_dist = compute_model_consensus_distance(round_models)
            consensus_distances[partition_name][round_num] = consensus_dist
            
            # Identify outliers
            outliers = identify_outlier_clients(round_models, threshold_std=2.0)
            outlier_clients[partition_name][round_num] = outliers
            
            # Compute distance to global
            global_distances = {}
            for client_id, local_state in round_models.items():
                distance = compute_layer_wise_l2_distance(local_state, global_state)
                avg_distance = np.mean(list(distance.values()))
                global_distances[client_id] = avg_distance
            
            weight_analysis[partition_name][round_num] = {
                'num_clients': len(round_models),
                'global_distances': global_distances,
                'avg_global_distance': np.mean(list(global_distances.values())) if global_distances else 0,
                'consensus_distance': consensus_dist
            }
            
            print(f"OK ({len(round_models)} clients)")
        except Exception as e:
            print(f"ERROR ({str(e)[:50]})")
            continue

print("\n✓ Weight analysis complete")

## Section 4: Visualizations

In [ ]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

# ========== Plot 1: Distance to Global Model Over Rounds ==========
ax1 = fig.add_subplot(gs[0, 0])

for partition_name, partition_data in weight_analysis.items():
    rounds = sorted(partition_data.keys())
    avg_distances = [partition_data[r]['avg_global_distance'] for r in rounds]
    label = partition_name.replace('_partitions', '').upper()
    ax1.plot(rounds, avg_distances, marker='o', linewidth=2.5, markersize=8, label=label, alpha=0.8)

ax1.set_xlabel('Communication Round', fontsize=12, fontweight='bold')
ax1.set_ylabel('Avg L2 Distance to Global Model', fontsize=12, fontweight='bold')
ax1.set_title('Model Convergence: Local-to-Global Distance', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11, loc='best')
ax1.grid(True, alpha=0.3)

# ========== Plot 2: Consensus Distance Evolution ==========
ax2 = fig.add_subplot(gs[0, 1])

for partition_name, partition_data in consensus_distances.items():
    rounds = sorted(partition_data.keys())
    consensus_dists = [partition_data[r] for r in rounds]
    label = partition_name.replace('_partitions', '').upper()
    ax2.plot(rounds, consensus_dists, marker='s', linewidth=2.5, markersize=8, label=label, alpha=0.8)

ax2.set_xlabel('Communication Round', fontsize=12, fontweight='bold')
ax2.set_ylabel('Model Consensus Distance', fontsize=12, fontweight='bold')
ax2.set_title('Client Model Homogeneity', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11, loc='best')
ax2.grid(True, alpha=0.3)

# ========== Plot 3: Similarity Matrix Heatmaps (Latest Round) ==========
ax3 = fig.add_subplot(gs[1, 0])

# Get latest round for each partition
latest_iid_round = max(similarity_matrices['iid_partitions'].keys()) if similarity_matrices['iid_partitions'] else None
if latest_iid_round and similarity_matrices['iid_partitions'][latest_iid_round][0] is not None:
    sim_matrix, client_ids = similarity_matrices['iid_partitions'][latest_iid_round]
    im = ax3.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax3.set_title(f'IID Similarity Matrix (Round {latest_iid_round})', fontsize=13, fontweight='bold')
    ax3.set_xlabel('Client ID', fontsize=11)
    ax3.set_ylabel('Client ID', fontsize=11)
    plt.colorbar(im, ax=ax3, label='Cosine Similarity')

# ========== Plot 4: Summary Statistics ==========
ax4 = fig.add_subplot(gs[1, 1])
ax4.axis('off')

summary_data = []
for partition_name in ['iid_partitions', 'non_iid_partitions']:
    partition_label = partition_name.replace('_partitions', '').upper()
    partition_data = weight_analysis[partition_name]
    
    if partition_data:
        all_distances = [d for data in partition_data.values() for d in data['global_distances'].values()]
        all_consensus = list(consensus_distances[partition_name].values())
        
        summary_data.append([
            partition_label,
            f"{np.mean(all_distances):.4f}" if all_distances else "N/A",
            f"{np.std(all_distances):.4f}" if all_distances else "N/A",
            f"{np.mean(all_consensus):.4f}" if all_consensus else "N/A"
        ])

table = ax4.table(
    cellText=summary_data,
    colLabels=['Partition', 'Avg Distance', 'Std Dev', 'Consensus'],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Style header
for i in range(4):
    table[(0, i)].set_facecolor('#40466e')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(summary_data) + 1):
    for j in range(4):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#f0f0f0')
        else:
            table[(i, j)].set_facecolor('#ffffff')

plt.suptitle('Weight Divergence Analysis\nFederated Learning with BiSeNetV2', 
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.savefig('weight_divergence_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualizations complete")

## Section 5: Findings & Interpretation

### Key Metrics Explained

1. **Distance to Global Model**: Average L2 norm between local and global weights
   - Measures how far clients drift from the aggregated model
   - Lower = better convergence towards consensus

2. **Consensus Distance**: Average pairwise distance between all client models
   - Measures client-to-client heterogeneity
   - Lower = more homogeneous client models

3. **Similarity Matrix**: Cosine similarity heatmap between all client pairs
   - Darker colors = more similar models
   - Patterns reveal client clustering or diversity

### Expected Patterns

**IID Partition** (Data balanced across clients):
- Smaller distance to global model (clients stay near consensus)
- Lower consensus distance (clients converge together)
- More uniform similarity matrix (all clients similar)

**Non-IID Partition** (Data heterogeneous across clients):
- Larger distance to global model (clients specialize on local data)
- Higher consensus distance (clients diverge to fit local distributions)
- Clustered similarity matrix (some clients very different)

### Connection to BN Divergence

🔗 **Hypothesis Link**:
- Larger weight divergence → Different feature distributions → Different BN statistics needed
- IID weight divergence < Non-IID weight divergence → IID BN divergence < Non-IID BN divergence
- Together, these metrics explain why Non-IID performance suffers under BN aggregation